**Esercizio**. Modificare la classe `LinearRegressionSGD` introducendo una condizione di arresto basata su *validation set*. Ovvero, durante l'addestramento, l'algoritmo non deve limitarsi a controllare la convergenza sul training set, ma deve monitorare anche l'errore su un insieme di validazione fornito separatamente.

In particolare, ad ogni epoca si richiede di:

- calcolare il valore della funzione di costo sia sul training set sia sul validation set;
- mantenere in memoria il miglior valore di errore sul validation set osservato fino a quel momento
- salvare i parametri del modello (pesi e intercetta) corrispondenti a tale valore minimo.

La procedura di addestramento deve essere interrotta nel caso in cui l'errore sul validation set non migliori per un numero prefissato di epoche consecutive (parametro `patience` da aggiungere all'elenco).

Al termine dell'addestramento, il modello deve:

- ripristinare i parametri corrispondenti al miglior valore di validation error;
- restituire tali parametri come soluzione finale.

Questa modifica consente di implementare una strategia di arresto, utile a prevenire fenomeni di overfitting, in quanto l'arresto dell'algoritmo avviene nel momento in cui le prestazioni su dati non visti (validation set) smettono di migliorare, anche se l'errore sul training set continua a diminuire.

In [2]:
from sklearn.datasets import make_regression
import numpy as np

In [6]:
class LinearRegressionSGD(object):
    def __init__(self, eta=0.001, n_iter=1000, tol=1e-5, batch_size=None):
        self.eta = eta
        self.n_iter = n_iter
        self.costi_ = []
        self.tol_ = tol
        self.batch_size_ = batch_size

    def net_input(self, X):
        return np.dot(X, self.w_[1:]) + self.w_[0]
        
    def predict(self, X):
        # Standardizza i dati usando media e std del training set
        X_std = (X - self.mean_) / self.std_
        
        return self.net_input(X_std)
    
    def get_coef(self):
        """
        Restituisce i coefficienti e l'intercetta del modello
        riferiti ai dati originali (non standardizzati).
        """

        # Coefficienti originali
        coef = self.w_[1:] / self.std_
        # Intercetta originale
        intercept = self.w_[0] - np.sum(self.w_[1:] * self.mean_ / self.std_)
        
        return intercept, coef
    
    def fit(self, X, y, X_val=None, y_val=None, patience=20):
        self.batch_size_ = X.shape[0] if self.batch_size_ is None else self.batch_size_

        # Standardizzazione (solo training)
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)

        X_std = (X - self.mean_) / self.std_

        if X_val is not None:
            X_val_std = (X_val - self.mean_) / self.std_

        # Inizializzazione pesi
        self.w_ = np.zeros(1 + X.shape[1])

        # Early stopping variables
        best_val_loss = np.inf
        best_w = self.w_.copy()
        patience_count = 0

        for epoca in range(self.n_iter):

            # shuffle
            shuffle_idxs = np.arange(X.shape[0])
            np.random.shuffle(shuffle_idxs)

            for b_idx in range(0, X.shape[0], self.batch_size_):
                X_b = X_std[shuffle_idxs[b_idx:b_idx+self.batch_size_]]
                y_b = y[shuffle_idxs[b_idx:b_idx+self.batch_size_]]

                output = self.net_input(X_b)
                errors = y_b - output

                self.w_[1:] += self.eta * X_b.T.dot(errors) / self.batch_size_
                self.w_[0] += self.eta * errors.sum() / self.batch_size_

            # training loss
            output = self.net_input(X_std)
            train_loss = mae(y, output)
            self.costi_.append(train_loss)

            # --- VALIDATION ---
            if X_val is not None:
                val_output = self.net_input(X_val_std)
                val_loss = mae(y_val, val_output)

                # controllo miglioramento
                if val_loss < best_val_loss - self.tol_:
                    best_val_loss = val_loss
                    best_w = self.w_.copy()
                    patience_count = 0
                else:
                    patience_count += 1

                # condizione di arresto
                if patience_count >= patience:
                    print(f"Stop dopo {epoca} passi")
                    break
            else:
                # vecchia condizione su training
                if len(self.costi_) >= 2:
                    if np.abs(self.costi_[-1] - self.costi_[-2]) < self.tol_:
                        break

        # ripristino migliori pesi
        if X_val is not None:
            self.w_ = best_w

        return self
    
def mae(y, z):
    '''
    Calcola l'errore medio assoluto
    '''
    return np.sum(np.abs(y-z))/y.shape[0]

In [34]:
X, y = make_regression(
    n_samples=1500,       # numero di campioni
    n_features=150,        # numero di feature
    noise=5.0,          # rumore da aggiungere ai target
    random_state=42      # per riproducibilità
)

n = X.shape[0]
train_size = 0.6
val_size = 0.2
np.random.seed(10)
train_idxs = np.random.choice(n, size = int(n*train_size), replace=False)

test_val_idxs =  np.setdiff1d(np.arange(n), train_idxs)

val_idxs = np.random.choice(test_val_idxs, size = int(n*val_size), replace=False)
test_idxs = np.setdiff1d(test_val_idxs, val_idxs)

X_train, X_val, X_test = X[train_idxs], X[val_idxs], X[test_idxs]
y_train, y_val, y_test = y[train_idxs], y[val_idxs], y[test_idxs]

linregSGD = LinearRegressionSGD(batch_size=100, n_iter=10000, tol=0.001)
linregSGD.fit(X_train, y_train, X_val=X_val, y_val=y_val, patience=5)

errorSGD = mae(y_train, linregSGD.predict(X_train))
print(f"Mean Absolute Error (MAE) train set: {errorSGD:.2f}. Iterazioni: {len(linregSGD.costi_)}")

errorSGD = mae(y_test, linregSGD.predict(X_test))
print(f"Mean Absolute Error (MAE) test set: {errorSGD:.2f}. Iterazioni: {len(linregSGD.costi_)}")

Stop dopo 912 passi
Mean Absolute Error (MAE) train set: 3.86. Iterazioni: 913
Mean Absolute Error (MAE) test set: 4.50. Iterazioni: 913


# DBSCAN

*Density-Based Spatial Clustering of Applications with Noise*. Algoritmo di clustering basato su densità.

M.Ester, H.Kriegel, J.Sander, X.Xu. *A Density-Based Algorithm for Discovering Clusters in Large Spatial Databases with Noise*. 1996.

Parametri:

- `min_pts`: numero minimo di punti per formare un cluster
- `eps`: distanza massima affinché due punti siano considerati vicini.

Numero di cluster non fissato a priori, viene deciso dall'algoritmo.

## Definizioni

- *Core point*: un punto che ha almeno `min_pts` in un suo intorno `eps`.
- *Border point*: un punto non core ma che è vicino ad un core
- *Noise point*: nè core, nè border

![image-2.png](08-01-dbscan_connessioni.png)


### Raggiungibilià

- Un punto `p` è *raggiungibile direttamente per densità* da `q` se `q` è un core e `p` è nel suo vicinato (fugura a sinistra). *In questo caso `q` appartiene ad un cluster*.
- Un punto `p` è *raggiungibile per densità* da `q` se esiste una catena di punti core a partire da `q` ciascuno raggiungibile direttamente per densità dal precedente (nella figura al centro, la catena contiene solo due punti core). *In questo caso tutti i core della catena sono punti di un unico cluster.*
- I punti `p` e `q` sono *connessi per densità*  se esiste un punto `u` da cu `p` e `q` sono raggiungibili per densità (figura a destra).

## Cluster

`C` è un cluster, rispetto `min_pts` e `eps` se è non vuoto e soddisfa le seguenti proprietà:
- Per ogni coppia `p` e `q`, se `q` è in `C` e `p` è raggiungibile per densità da `q`, anche `p` appartiene a `C`.
- Ogni coppia `p` e `q` in `C` deve essere connessa per densità. (Ovvero solo i punti raggiungibili per densità da un core in `C` sono in `C`)

Dalla definizione, i border possono appartenere a più cluster perché raggiungibili per densità da più core non vicini. L'algoritmo ne sceglierà uno e sarà il primo da cui viene raggiunto.

## L'Algoritmo

- INPUT: un insieme `D` di `n` punti, `eps`, `min_pts`
- OUTPUT: per ogni punto `p` in `D`, un valore `label[p]` in `{Noise, 0, 1, ...}`

* Inizializzazione: `label[p] ← non_visitato` per tutti i punti `p`
* Per ogni punto `p` in `D` etichettato `non_visitato`:
    1. Sia `N` l'insieme dei punti a distanza `≤ eps` da `p`
    2. Se `|N| < min_pts`, non è un core point, ma potrebbe diventare border. Per il momento
        - `label[p] ← Noise`
    3. Altrimenti `p` è un core di un nuovo cluster `c`, quindi
        * `label[p] ← c`
        * `S ← N - {p}`
        * Prosegui prelevando punti `q` da `S` fino ad esaurimento di quest'ultimo
            - Se `q` è già in un cluster, scartalo (se `label[q]` è un numero)
            - `label[q] ← c` anche se valeva `Noise`
            - Se `q` è un core point
                - Definisci `N` come l'insieme dei vicini di `q`
                - `S ← S ∪ (N - {q})`